combined PAC analysis. joins the temporal classifier (nb 05, all_bylines_bands.csv) and the content classifier (nb 06, content_classifier_per_advertiser.csv) per-advertiser, computes a combined PAC-likeness score, and produces two tables for the report:

1. top 20 advertisers by total spend, with their three metrics (band / EC index, mean election_prob inside campaign, content shift) and the combined PAC score.
2. top 10 PACs (highest combined score), with their top 3 ads by spend each.

the combined score is `pac_score = ec_index * mean_prob_inside` --- both metrics are bounded [0, 1], so the product is a strict-AND composite: high only if temporal concentration AND election-content density are high together.

setup.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, coalesce, substring, first,
    sum as spark_sum, count as spark_count,
    row_number, desc,
)
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

spark = SparkSession.builder \
    .appName('FB_API_combined_pacs') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

paths.

In [ ]:
V3_PATH        = '/user/s3348393/main/preprocessing/v3/parquet'
BANDS_CSV      = '../data/all_bylines_bands.csv'
CONTENT_CSV    = '../data/content_classifier_per_advertiser.csv'

TOP_20_OUT_CSV     = '../data/top_20_with_metrics.csv'
TOP_PACS_OUT_CSV   = '../data/top_10_pacs_with_top_ads.csv'

TOP_N_SPENDERS    = 20
TOP_N_PACS        = 10
TOP_N_ADS_PER_PAC = 3

load temporal and content scores per advertiser, join them, compute the combined PAC score.

In [ ]:
bands_df   = pd.read_csv(BANDS_CSV)
content_df = pd.read_csv(CONTENT_CSV)

print(f'temporal classifier rows: {len(bands_df):,}')
print(f'content classifier rows:  {len(content_df):,}')

# Inner-join on bylines --- only advertisers that survived both classifiers
metrics = (bands_df[['bylines', 'band', 'persistence', 'ec_index']]
    .merge(
        content_df[['bylines', 'mean_prob_inside', 'shift',
                    'mean_election_prob_weighted', 'total_spend']],
        on='bylines', how='inner'
    ))

# Combined PAC-likeness score: strict-AND multiplicative composite
metrics['pac_score'] = metrics['ec_index'] * metrics['mean_prob_inside']

print(f'\njoined metrics: {len(metrics):,} advertisers')
print(f'pac_score range: {metrics["pac_score"].min():.3f} - {metrics["pac_score"].max():.3f}')

table 1: top 20 advertisers by total spend, with their three metrics and the combined PAC score.

In [ ]:
top20_metrics = (metrics
    .sort_values('total_spend', ascending=False)
    .head(TOP_N_SPENDERS)
    .reset_index(drop=True))

display_cols = ['bylines', 'total_spend', 'band',
                'ec_index', 'mean_prob_inside', 'shift', 'pac_score']

print(f'Top {TOP_N_SPENDERS} advertisers by total spend:')
display(top20_metrics[display_cols])

top20_metrics[display_cols].to_csv(TOP_20_OUT_CSV, index=False)
print(f'\nWrote {TOP_20_OUT_CSV}')

table 2: top 10 PACs by combined score, with each PAC's top 3 ads by spend.

In [ ]:
# Identify the top 10 PACs by combined PAC score
top_pacs = (metrics
    .sort_values('pac_score', ascending=False)
    .head(TOP_N_PACS)
    .reset_index(drop=True))
top_pac_bylines = top_pacs['bylines'].tolist()

print(f'Top {TOP_N_PACS} PACs by pac_score:')
display(top_pacs[['bylines', 'pac_score', 'band', 'ec_index',
                  'mean_prob_inside', 'shift', 'total_spend']])

# Pull each PAC's top 3 ads from v3 (by total spend per unique creative title)
def first_non_empty(col_name):
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")

ad_title = coalesce(
    first_non_empty('creative_link_titles'),
    substring(first_non_empty('creative_bodies'), 1, 80),
)

v3 = spark.read.parquet(V3_PATH).filter(col('match_type').isNull())

ad_spend = (v3
    .filter(col('bylines').isin(top_pac_bylines) & col('spend_mid').isNotNull())
    .withColumn('ad_title', ad_title)
    .filter(col('ad_title').isNotNull())
    .groupBy('bylines', 'ad_title')
    .agg(
        spark_sum('spend_mid').alias('total_spend'),
        spark_count('*').alias('n_runs'),
        first('ad_snapshot_url').alias('example_url'),
    ))

w = Window.partitionBy('bylines').orderBy(desc('total_spend'))
top_pac_ads = (ad_spend
    .withColumn('rank', row_number().over(w))
    .filter(col('rank') <= TOP_N_ADS_PER_PAC)
    .orderBy('bylines', 'rank')
    .toPandas())

# Preserve PAC ranking order
top_pac_ads['_byline_order'] = top_pac_ads['bylines'].map(
    {b: i for i, b in enumerate(top_pac_bylines)})
top_pac_ads = (top_pac_ads
    .sort_values(['_byline_order', 'rank'])
    .drop(columns='_byline_order')
    .reset_index(drop=True))

print(f'\n{len(top_pac_ads)} rows: top {TOP_N_ADS_PER_PAC} ads for each of the top {TOP_N_PACS} PACs.')

visualisations. (1) a PAC-likeness scatter showing where every advertiser sits in temporal-classifier $\times$ content-classifier space, with the top 20 spenders highlighted and the top 10 PACs in bold. (2) a horizontal bar chart of the top 10 PACs ranked by combined PAC score.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

BAND_COLORS = {
    'election_only': '#1f77b4',
    'transitional':  '#ff7f0e',
    'ongoing':       '#2ca02c',
    'off_campaign':  '#d62728',
}

# === Viz 1: PAC-likeness scatter (temporal x content) ===
# Background: all advertisers as small grey dots.
# Foreground: top 20 spenders sized by log spend, coloured by band.
# Top 10 PACs labelled in bold.

fig, ax = plt.subplots(figsize=(11, 9))

ax.scatter(metrics['ec_index'], metrics['mean_prob_inside'],
           s=20, c='lightgray', alpha=0.5, edgecolor='none', zorder=1)

top20_colours = top20_metrics['band'].map(BAND_COLORS).fillna('#888888')
spend_sizes = (np.log10(top20_metrics['total_spend'].fillna(1)) - 4).clip(lower=0.5) * 80

ax.scatter(top20_metrics['ec_index'], top20_metrics['mean_prob_inside'],
           s=spend_sizes, c=top20_colours, alpha=0.85,
           edgecolor='black', linewidth=0.5, zorder=3)

top_pac_set = set(top_pacs['bylines'])
texts = []
for _, row in top20_metrics.iterrows():
    is_pac = row['bylines'] in top_pac_set
    texts.append(ax.text(
        row['ec_index'], row['mean_prob_inside'],
        ' ' + row['bylines'][:30],
        fontsize=8 if is_pac else 7,
        fontweight='bold' if is_pac else 'normal',
        alpha=0.95 if is_pac else 0.65,
        zorder=5 if is_pac else 4,
    ))

try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=0.6))
except ImportError:
    pass

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)

ax.text(0.98, 0.98, 'high temporal\nhigh content\n(strong PAC)',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=9, alpha=0.45, fontstyle='italic')
ax.text(0.02, 0.02, 'low temporal\nlow content\n(not PAC-like)',
        transform=ax.transAxes, ha='left', va='bottom',
        fontsize=9, alpha=0.45, fontstyle='italic')

band_handles = [
    Line2D([0], [0], marker='o', linestyle='', color='w',
           markerfacecolor=c, markeredgecolor='black', markersize=10, label=b)
    for b, c in BAND_COLORS.items()
]
ax.legend(handles=band_handles, loc='lower right', fontsize=9, title='Band')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('EC index (temporal classifier - higher = more election-window concentrated)')
ax.set_ylabel('Mean election_prob inside campaign (content classifier)')
ax.set_title('PAC-likeness space: top 20 spenders highlighted, top 10 PACs in bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# === Viz 2: Top 10 PACs by pac_score (horizontal bar) ===

fig, ax = plt.subplots(figsize=(11, 6))

top_pacs_sorted = top_pacs.sort_values('pac_score', ascending=True).reset_index(drop=True)
y_pos = np.arange(len(top_pacs_sorted))
colours = top_pacs_sorted['band'].map(BAND_COLORS).fillna('#888888')

ax.barh(y_pos, top_pacs_sorted['pac_score'], color=colours,
        edgecolor='black', linewidth=0.5, alpha=0.85)

for i, (_, row) in enumerate(top_pacs_sorted.iterrows()):
    ax.text(row['pac_score'] + 0.005, i,
            f'  ${row["total_spend"]/1000:,.0f}k   ec={row["ec_index"]:.2f}   p_in={row["mean_prob_inside"]:.2f}',
            va='center', fontsize=8, alpha=0.75)

ax.set_yticks(y_pos)
ax.set_yticklabels([b[:35] for b in top_pacs_sorted['bylines']])
ax.set_xlim(0, top_pacs_sorted['pac_score'].max() * 1.85)
ax.set_xlabel('PAC score (EC index x mean election_prob inside campaign)')
ax.set_title('Top 10 Australian PACs by combined classifier score')

band_handles = [
    Line2D([0], [0], marker='s', linestyle='', color='w',
           markerfacecolor=c, markeredgecolor='black', markersize=10, label=b)
    for b, c in BAND_COLORS.items()
]
ax.legend(handles=band_handles, loc='lower right', fontsize=9, title='Band')

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

per-PAC display and export.

In [ ]:
for byline in top_pac_bylines:
    sub = top_pac_ads[top_pac_ads['bylines'] == byline]
    pac_row = top_pacs[top_pacs['bylines'] == byline].iloc[0]
    print(f'\n=== {byline}  (pac_score={pac_row["pac_score"]:.3f}, '
          f'${pac_row["total_spend"]:,.0f} total, band={pac_row["band"]}) ===')
    if sub.empty:
        print('(no ads matched filter)')
        continue
    display(sub[['rank', 'ad_title', 'total_spend', 'n_runs']])

# Join PAC-level scores onto the ad-level rows for the export
top_pac_ads_export = top_pac_ads.merge(
    top_pacs[['bylines', 'pac_score', 'band']],
    on='bylines', how='left')

out_cols = ['bylines', 'pac_score', 'band', 'rank', 'ad_title',
            'total_spend', 'n_runs', 'example_url']
top_pac_ads_export[out_cols].to_csv(TOP_PACS_OUT_CSV, index=False)
print(f'\nWrote {TOP_PACS_OUT_CSV}')